# 01 种子入库

把 v2 种子（`data/raw/e09-paper-knowledge/seeds/`）写进 neo4j-e09，这是 e09 第一次写图。数据模型见 `docs/designs/v2/graph_model_v2.md`，种子格式见种子目录的 `README.md`。

写法沿用 e08（`04_neo4j`、`06_extract_prepare`）：读文件 → plan（只查不写）→ apply（单事务）→ 复核，重跑无副作用。和 e08 不同的是**身份怎么判断**：

- e08 按"主 Label + `name`"判断节点是否已存在；
- v2 按 NameKey 精确键（`intents_decompose.md` §3.2 的 D3）与 `identifiers`（`graph_model_v2.md` §2.4）解析。每条匹配规则都有四种结果：唯一命中、多重命中、无命中、数据冲突。plan 把冲突和查重候选显式列出来，第 5 节用不写库的小批次逐一演示。

plan 与 apply 对应 `Commit(delta, mode: dry_run | apply)` 的两种模式。这里只是种子的最小写入路径，不是入库表单的编译器：种子没有 Content、`FROM` 和 ResultUnit。

## 1. 预设参数

路径与连接参数来自 `e09.config`，与 `make check` 共用。

In [ ]:
import pandas as pd
import yaml

from e09.config import NEO4J_AUTH, NEO4J_DB, NEO4J_URI, SEEDS

SEED_FILES = sorted(SEEDS.glob("*.yml"))
RESET = False   # True：清空整个库（DETACH DELETE 全部节点）后从空图重演；仅限 neo4j-e09 只放种子时使用
print(NEO4J_URI, [p.name for p in SEED_FILES])

## 2. 读种子，做与库无关的检查

`check_seed` 只看文件本身，不查库：

- `labels` 恰好两个：主 Label（Entity / Concept）与 kind，kind 在种子允许的范围内（种子 `README.md`）；
- 不自带 `id`（入库时分配）；必填 `name`，Concept 必填 `definition`，Entity 必填 `description`；Metric 的 `direction` 为 `lower` / `higher`；
- `identifiers` 每项是已声明命名空间的 `"<namespace>:<value>"`；唯一命名空间（`arxiv`、`doi`、`s2`）批内不能重复；
- `name` 与 `aliases` 各算一个精确键 `规范化名称|kind|scope`，批内同一个键不能注册给两个节点；
- 关系的端点在本文件内，类型与端点的主 Label、kind 符合 `graph_model_v2.md` §6。

精确键的规范化是 `name-key-v1`（`intents_decompose.md` §3.2）：NFC；由 Unicode White_Space 字符组成的连续空白折叠为一个空格，再去掉首尾空白；保留大小写与标点。所以 `BM-25` 与 `BM25`、`BERT` 与 `bert` 是不同的键。种子都是全局 scope；切分（Split）的 scope 是父数据集，种子里没有。

`aliases` 是表单字段，入库时注册为 NameKey，不写进对象；`sources`、`verified` 只留在种子文件里。

工具函数（之后会提取出来作为脚本使用）

In [ ]:
import re
import unicodedata

KINDS = {"Entity": ["Paper", "Dataset", "Split", "Code", "Model", "Benchmark"],         # graph_model_v2 §2.1
         "Concept": ["Method", "Task", "Metric", "Protocol", "Issue", "Proposition"]}   # §3.1
SEED_KINDS = {"Entity": {"Dataset", "Model", "Benchmark"}, "Concept": {"Task", "Method", "Metric"}}   # 种子只预置这些
REQUIRED = {"Entity": "description", "Concept": "definition"}
NAMESPACES = {"arxiv": True, "doi": True, "s2": True, "url": False}   # 命名空间 -> 是否唯一（§2.4）
REL_RULES = {   # 种子用到的关系 -> (起点主 Label, 终点主 Label, 终点 kind 限制)；§6
    "BROADER": ("Concept", "Concept", None), "OVERLAPS_WITH": ("Concept", "Concept", None),
    "PART_OF": ("Entity", "Entity", None), "FOR_TASK": ("Entity", "Concept", "Task")}
SYMMETRIC = {"OVERLAPS_WITH"}   # 语义对称，不分方向
FORM_ONLY = {"aliases"}         # 表单字段：注册为 NameKey，不写进对象

# name-key-v1：White_Space 是 Unicode 属性，与 Python 的 str.isspace 不完全一致，这里显式列出
WHITE_SPACE = re.compile("[\u0009-\u000d \u0085   -     　]+")
# 设计要求 Unicode 15.1；NFC 用的是 Python 自带的 unicodedata，版本如实记进 normalizer_ref
NORMALIZER = f"name-key-v1@unicode-{unicodedata.unidata_version}"


def normalize(s: str) -> str:
    return WHITE_SPACE.sub(" ", unicodedata.normalize("NFC", s)).strip()


def name_key(raw: str, kind: str, scope: str = "global") -> str:
    """NameKey.key：规范化名称、kind、scope 拼成的单个字符串，唯一约束作用在它上面。"""
    return f"{normalize(raw)}|{kind}|{scope}"


def scalar_or_list(v) -> bool:
    """Neo4j 属性只能是标量或标量列表。"""
    return isinstance(v, (str, int, float, bool)) or (isinstance(v, list) and all(isinstance(x, (str, int, float, bool)) for x in v))


def check_seed(raw: dict, name: str) -> dict:
    """返回 {name, nodes: {ref: node}, rels: [(from, type, to)]}；node 带 family、kind、写进图的 props 与精确键 keys。"""
    nodes, rels, seen_rels, key_owner, uniq_owner = {}, [], set(), {}, {}
    for n in raw.get("nodes") or []:
        ref, props = n["ref"], dict(n.get("properties") or {})
        assert ref not in nodes, f"{name}: 批内重复 ref {ref}"
        assert len(n["labels"]) == 2, f"{name}: {ref} 需要恰好两个 Label（主 Label + kind），实际是 {n['labels']}"
        family, kind = n["labels"]
        assert kind in SEED_KINDS.get(family, ()), f"{name}: {ref} 的 {n['labels']} 不在种子范围 {SEED_KINDS}"
        assert "id" not in props, f"{name}: {ref} 不应自带 id，id 由入库分配"
        assert props.get("name") and props.get(REQUIRED[family]), f"{name}: {ref} 缺 name 或 {REQUIRED[family]}"
        if kind == "Metric":
            assert props.get("direction") in ("lower", "higher"), f"{name}: {ref} 的 direction 应为 lower / higher"
        bad = [k for k, v in props.items() if not scalar_or_list(v)]
        assert not bad, f"{name}: {ref} 的属性 {bad} 不是标量或标量列表"
        for i in props.get("identifiers", []):
            ns, sep, value = i.partition(":")
            assert sep and value and ns in NAMESPACES, f"{name}: {ref} 的标识 {i!r} 不是已声明命名空间的 <namespace>:<value>"
            if NAMESPACES[ns]:
                assert i not in uniq_owner, f"{name}: 唯一标识 {i} 同时出现在 {uniq_owner[i]} 与 {ref}"
                uniq_owner[i] = ref
        keys = {}   # key -> 原字符串；同一节点的两个称呼规范化后相同，只注册一次
        for s in [props["name"], *props.get("aliases", [])]:
            assert "|" not in normalize(s), f"{name}: {ref} 的称呼 {s!r} 含键分隔符 |"
            k = name_key(s, kind)
            assert key_owner.get(k, ref) == ref, f"{name}: 精确键 {k!r} 同时注册给 {key_owner[k]} 与 {ref}"
            key_owner[k] = ref
            keys.setdefault(k, s)
        nodes[ref] = {"labels": n["labels"], "family": family, "kind": kind, "keys": keys,
                      "props": {k: v for k, v in props.items() if k not in FORM_ONLY}}
    for r in raw.get("relationships") or []:
        a, t, c = r["from"], r["type"], r["to"]
        assert a in nodes and c in nodes, f"{name}: 关系 {a} -{t}-> {c} 的端点不在本文件"
        assert t in REL_RULES, f"{name}: 关系类型 {t!r} 不在 {sorted(REL_RULES)}"
        fa, fc, kc = REL_RULES[t]
        ok = (nodes[a]["family"], nodes[c]["family"]) == (fa, fc) and kc in (None, nodes[c]["kind"])
        if t == "BROADER":
            ok = ok and nodes[a]["kind"] == nodes[c]["kind"]
        assert ok, f"{name}: {a} {nodes[a]['labels']} -{t}-> {c} {nodes[c]['labels']} 不符合 {t} 的端点定义"
        ident = (min(a, c), t, max(a, c)) if t in SYMMETRIC else (a, t, c)
        assert ident not in seen_rels, f"{name}: 批内重复关系 {a} -{t}- {c}"
        seen_rels.add(ident)
        rels.append((a, t, c))
    return {"name": name, "nodes": nodes, "rels": rels}


def load_seed(path) -> dict:
    return check_seed(yaml.safe_load(path.read_text(encoding="utf-8")), path.name)

In [ ]:
seeds = {p.name: load_seed(p) for p in SEED_FILES}
print(NORMALIZER)
pd.DataFrame([{"file": name, "nodes": len(b["nodes"]), "name_keys": sum(len(n["keys"]) for n in b["nodes"].values()),
               "relationships": len(b["rels"])} for name, b in seeds.items()]).set_index("file")

## 3. 连接 neo4j-e09，建约束

neo4j-e08 与 neo4j-e09 端口相同，写之前先用 `env_check.check_graph()` 确认没有连到 v1 库。

约束全部 `IF NOT EXISTS`，重跑无副作用：

- `Entity.id`、`Concept.id` 唯一；
- `NameKey.key` 唯一。D3 的"写入时唯一"由这条约束兜底，这也是 NameKey 不存成对象上 `aliases` 列表的原因：唯一约束作用于整个属性值，不作用于列表元素（`graph_model_v2.md` 第 5 节）。

唯一命名空间的 `identifiers` 是对象上的字符串列表，数据库约束管不到，由 plan 检查、apply 在同一事务内复查。

In [ ]:
from neo4j import GraphDatabase, RoutingControl

from e09.env_check import check_graph

assert not check_graph(), "不是 v2 库，停止"

# 空库时查不存在的属性键或 Label 会收到 UNRECOGNIZED 类提示，关掉
driver = GraphDatabase.driver(NEO4J_URI, auth=NEO4J_AUTH, notifications_disabled_classifications=["UNRECOGNIZED"])
driver.verify_connectivity()


def q(cypher: str, **params) -> list[dict]:
    """只读查询的简写。"""
    records, _, _ = driver.execute_query(cypher, params, database_=NEO4J_DB, routing_=RoutingControl.READ)
    return [r.data() for r in records]


SCHEMA = [
    "CREATE CONSTRAINT entity_id IF NOT EXISTS FOR (n:Entity) REQUIRE n.id IS UNIQUE",
    "CREATE CONSTRAINT concept_id IF NOT EXISTS FOR (n:Concept) REQUIRE n.id IS UNIQUE",
    "CREATE CONSTRAINT namekey_key IF NOT EXISTS FOR (k:NameKey) REQUIRE k.key IS UNIQUE",
]
for stmt in SCHEMA:
    driver.execute_query(stmt, database_=NEO4J_DB)


def library_status():
    total = q("MATCH (n) RETURN count(n) AS c")[0]["c"]   # 每个节点带两个 Label，不能按 Label 计数相加
    kinds = q("MATCH (n) RETURN labels(n)[0] AS a, labels(n)[1] AS b, count(*) AS n")
    rels = q("MATCH ()-[r]->() RETURN type(r) AS type, count(*) AS n ORDER BY type")
    print(f"节点 {total}，关系 {sum(r['n'] for r in rels)}")
    if kinds:   # Neo4j 不保证 labels(n) 的顺序，这里按主 Label 摆正
        rows = [{"label": ":".join(sorted([r["a"], r["b"]] if r["b"] else [r["a"]],
                                          key=lambda l: l not in ("Entity", "Concept"))), "n": r["n"]} for r in kinds]
        display(pd.DataFrame(rows).groupby("label").sum().T)
    if rels:
        display(pd.DataFrame(rels).set_index("type").T)


if RESET:
    deleted = driver.execute_query("MATCH (n) DETACH DELETE n", database_=NEO4J_DB).summary.counters.nodes_deleted
    print(f"RESET：删除 {deleted} 个节点")

display(pd.DataFrame(q("SHOW CONSTRAINTS YIELD name, labelsOrTypes, properties")))
library_status()

## 4. 写入：plan → apply → 复核

**`plan_seed`**（对应 `Commit` 的 `dry_run`）只查不写，对每个节点做写入模式的解析：

1. **对象身份看 `name` 的精确键。** 命中某个对象就是该对象（D3 唯一键直接 resolved），属性有差异的列为"更新"；未命中就是新对象。
2. **每个精确键（`name` 与各 alias）分四种：**
   - 未注册 → 新键；
   - 已注册给同一对象 → 不变（同键同对象的重复注册是幂等的）；
   - 已注册给别的对象 → **冲突**；
   - 已隔离为 `ambiguous` → **冲突**，暂停精确解析。

   名称未命中、某个 alias 却指向已有对象 X，也属于冲突：它是 X 的新名称，还是另一个对象，要交给外部判断。
3. **`identifiers` 按命名空间处理（§2.4）：**
   - 唯一命名空间被别的对象占用 → **冲突**。
   - 非唯一的 `url` 被别的对象（库里的，或本批里的）同时持有 → **查重候选**，不阻止写入。这和 Resolve 写入模式的约定一致：多重命中只供外部判断"新对象是其中之一，还是同一仓库里的另一个对象"，不直接复用。种子里同一仓库下的多个数据集是作者有意分开建的，所以查重候选在这里只列出、不处理。
4. 关系以 `(from, type, to)` 为身份；两端都已在库里时查一次是否已存在，语义对称的不分方向。

**`apply_plan`**（对应 `apply`）：有冲突就整批拒绝；没有变化就跳过。否则在**一个写事务**里完成以下步骤，中途出错整批回滚：

- 建新对象；
- 已有对象 `SET n += props`；
- 注册 NameKey；
- 复查唯一标识；
- 建关系。

NameKey 先 `MERGE`，再在同一事务里确认这个键只指向本对象、且仍是 `active`，确保唯一检查与写入原子执行。写完再 plan 一次，必须没有任何待写内容。

更新语义和 e08 一样是 `SET n += props`：从种子里删掉一个属性或 alias **不会**删掉库里的值，NameKey 的撤销与改指向（对象合并，Q4）不在这里。

**id** 形如 `<kind 前缀>_<4 位序号>`（如 `task_0001`、`dataset_0001`），由 `IdAllocator` 从图中已用的最大序号起顺序分配；分配后写入失败只会留下空号，不会重号。

工具函数（之后会提取出来作为脚本使用）

In [ ]:
from collections import defaultdict
from dataclasses import dataclass, field

PREFIX = {"Paper": "paper", "Dataset": "dataset", "Split": "split", "Code": "code", "Model": "model",
          "Benchmark": "bench", "Method": "method", "Task": "task", "Metric": "metric", "Protocol": "protocol",
          "Issue": "issue", "Proposition": "prop"}
FAMILY = {k: f for f, ks in KINDS.items() for k in ks}


class IdAllocator:
    """起点是图中该前缀已用的最大序号；同一次运行内共用一个实例。"""

    def __init__(self):
        self.last = {}
        for kind, prefix in PREFIX.items():
            rows = q(f"MATCH (n:{FAMILY[kind]}:{kind}) WHERE n.id STARTS WITH $p "
                     "RETURN max(toInteger(substring(n.id, size($p)))) AS m", p=prefix + "_")
            self.last[kind] = rows[0]["m"] or 0

    def new(self, kind: str) -> str:
        self.last[kind] += 1
        return f"{PREFIX[kind]}_{self.last[kind]:04d}"


@dataclass
class Plan:
    batch: dict
    target: dict = field(default_factory=dict)       # ref -> 已有对象的 id；不在其中的是新对象
    new_nodes: list = field(default_factory=list)
    upd_nodes: dict = field(default_factory=dict)    # ref -> {字段: (库里的值, 种子的值)}
    same_nodes: list = field(default_factory=list)
    new_keys: list = field(default_factory=list)     # (ref, 原字符串, key)
    same_keys: int = 0
    new_rels: list = field(default_factory=list)
    same_rels: list = field(default_factory=list)
    conflicts: list = field(default_factory=list)    # 拒绝写入的原因
    candidates: list = field(default_factory=list)   # 非唯一标识的查重候选：不阻止写入，交外部判断

    @property
    def empty(self) -> bool:
        return not (self.new_nodes or self.upd_nodes or self.new_keys or self.new_rels)


def diff_props(old: dict, new: dict) -> dict:
    """种子里写了、且和库里不同的字段。"""
    return {k: (old.get(k), v) for k, v in new.items() if old.get(k) != v}


def plan_seed(b: dict) -> Plan:
    p = Plan(b)
    nodes = b["nodes"]

    # 1–2. 精确键：一次查出本批全部键的注册情况
    hits = defaultdict(list)   # key -> [{status, id}]；正常至多一个对象
    for r in q("MATCH (k:NameKey) WHERE k.key IN $keys OPTIONAL MATCH (k)-[:NAMES]->(o) "
               "RETURN k.key AS key, k.status AS status, o.id AS id",
               keys=[k for n in nodes.values() for k in n["keys"]]):
        hits[r["key"]].append(r)
    for ref, n in nodes.items():
        owners = sorted({h["id"] for h in hits[name_key(n["props"]["name"], n["kind"])] if h["id"]})
        if len(owners) == 1:
            p.target[ref] = owners[0]
        for k, raw in n["keys"].items():
            hs = hits[k]
            held = sorted({h["id"] for h in hs if h["id"]})
            if any(h["status"] != "active" for h in hs):
                p.conflicts.append({"ref": ref, "rule": "key-ambiguous", "what": k, "holders": held})
            elif not held:
                p.new_keys.append((ref, raw, k))
            elif held == [p.target.get(ref)]:
                p.same_keys += 1
            else:
                p.conflicts.append({"ref": ref, "rule": "key-taken", "what": k, "holders": held})

    # 已有对象：Label 必须一致，属性差异列为更新
    db = {r["id"]: r for r in q("MATCH (n) WHERE (n:Entity OR n:Concept) AND n.id IN $ids "
                                "RETURN n.id AS id, labels(n) AS labels, properties(n) AS props",
                                ids=sorted(set(p.target.values())))}
    for ref, nid in p.target.items():
        if set(db[nid]["labels"]) != set(nodes[ref]["labels"]):
            p.conflicts.append({"ref": ref, "rule": "label-mismatch", "what": nid, "holders": db[nid]["labels"]})
        elif d := diff_props(db[nid]["props"], nodes[ref]["props"]):
            p.upd_nodes[ref] = d
        else:
            p.same_nodes.append(ref)
    p.new_nodes = [ref for ref in nodes if ref not in p.target]

    # 3. identifiers：库里的持有者，加上本批其他节点
    by_ident = defaultdict(list)
    for ref, n in nodes.items():
        for i in n["props"].get("identifiers", []):
            by_ident[i].append(ref)
    holders = defaultdict(list)
    for r in q("MATCH (o) WHERE (o:Entity OR o:Concept) AND any(i IN o.identifiers WHERE i IN $ids) "
               "UNWIND [i IN o.identifiers WHERE i IN $ids] AS i RETURN i, o.id AS id, o.name AS name",
               ids=list(by_ident)):
        holders[r["i"]].append(r)
    for i, refs in by_ident.items():
        unique = NAMESPACES[i.partition(":")[0]]
        for ref in refs:
            in_graph = [f"{h['id']} {h['name']}" for h in holders[i] if h["id"] != p.target.get(ref)]
            if unique and in_graph:
                p.conflicts.append({"ref": ref, "rule": "identifier-taken", "what": i, "holders": in_graph})
            elif not unique and ref not in p.target:   # 写入模式下，只有新对象需要查重
                in_batch = [r for r in refs if r != ref and r not in p.target]
                if in_graph or in_batch:
                    p.candidates.append({"identifier": i, "ref": ref, "in_graph": in_graph, "in_batch": in_batch})

    # 4. 关系：两端都已在库里才可能已存在
    known = [{"a": p.target[a], "t": t, "c": p.target[c], "k": [a, t, c]}
             for a, t, c in b["rels"] if a in p.target and c in p.target]
    found = {tuple(r["k"]) for r in q(
        """UNWIND $rows AS row
           MATCH (a {id: row.a})-[r]-(c {id: row.c})
           WHERE type(r) = row.t AND (startNode(r) = a OR row.t IN $sym)
           RETURN DISTINCT row.k AS k""", rows=known, sym=sorted(SYMMETRIC))}
    for key in b["rels"]:
        (p.same_rels if key in found else p.new_rels).append(key)
    return p


def short(v, n=60):
    s = str(v).replace("\n", " ")
    return s if len(s) <= n else s[:n] + "…"


def show_plan(p: Plan):
    print(f"== {p.batch['name']} ==")
    print(f"对象    新建 {len(p.new_nodes)} · 更新 {len(p.upd_nodes)} · 不变 {len(p.same_nodes)}")
    print(f"精确键  新注册 {len(p.new_keys)} · 已注册 {p.same_keys}")
    print(f"关系    新建 {len(p.new_rels)} · 已存在 {len(p.same_rels)}")
    for ref, d in p.upd_nodes.items():
        print(f"更新 {ref} → {p.target[ref]}：")
        for k, (old, new) in d.items():
            print(f"    {k}: {short(old)} → {short(new)}")
    if p.candidates:
        print(f"查重候选（非唯一标识，不阻止写入）{len(p.candidates)} 条：")
        display(pd.DataFrame(p.candidates).groupby("identifier").agg(
            new=("ref", list), in_graph=("in_graph", lambda s: sorted({x for v in s for x in v}))))
    if p.conflicts:
        print(f"冲突 {len(p.conflicts)} 处 → apply 会拒绝整批：")
        display(pd.DataFrame(p.conflicts))
    elif p.empty:
        print("→ 与库内容一致，无需写入")

In [ ]:
def apply_plan(p: Plan, ids: IdAllocator) -> dict:
    """写入一批；返回 ref -> id。有冲突时不写，返回空 dict。"""
    b = p.batch
    if p.conflicts:
        print(f"{b['name']}：{len(p.conflicts)} 处冲突，拒绝写入（整批不写）")
        return {}
    if p.empty:
        print(f"{b['name']}：无变化，跳过写入")
        return dict(p.target)

    nodes = b["nodes"]
    ref2id = dict(p.target) | {ref: ids.new(nodes[ref]["kind"]) for ref in p.new_nodes}
    source = f"seed:{b['name']}"

    def work(tx):
        # 新对象：按 Label 组合分组
        groups = defaultdict(list)
        for ref in p.new_nodes:
            groups[tuple(nodes[ref]["labels"])].append({"id": ref2id[ref], **nodes[ref]["props"]})
        for labels, rows in groups.items():
            c = tx.run(f"UNWIND $rows AS row CREATE (n:{':'.join(labels)}) SET n = row RETURN count(n) AS c",
                       rows=rows).single()["c"]
            assert c == len(rows)
        # 已有对象：种子写了的属性覆盖，没写的保留
        for ref in p.upd_nodes:
            fam = nodes[ref]["family"]
            tx.run(f"MATCH (n:{fam} {{id: $id}}) SET n += $props", id=ref2id[ref], props=nodes[ref]["props"])
        # NameKey：MERGE 后在同一事务里确认该键只指向本对象且仍为 active，否则抛错、整批回滚
        groups = defaultdict(list)
        for ref, raw, k in p.new_keys:
            n = nodes[ref]
            groups[n["family"]].append({"id": ref2id[ref], "key": k, "props": {
                "key": k, "normalized": normalize(raw), "raw": raw, "kind": n["kind"], "scope": "global",
                "normalizer_ref": NORMALIZER, "status": "active", "registered_from": source, "registered_by": "seed"}})
        for fam, rows in groups.items():
            res = tx.run(f"""UNWIND $rows AS row
                             MATCH (o:{fam} {{id: row.id}})
                             MERGE (k:NameKey {{key: row.key}}) ON CREATE SET k = row.props
                             MERGE (k)-[:NAMES]->(o)
                             WITH k, row
                             MATCH (k)-[:NAMES]->(x)
                             RETURN row.key AS key, row.id AS id, k.status AS status, collect(x.id) AS owners""",
                         rows=rows).data()
            bad = [r for r in res if r["owners"] != [r["id"]] or r["status"] != "active"]
            if len(res) != len(rows) or bad:
                raise RuntimeError(f"NameKey 写入冲突（整批回滚）：{bad or '有对象没匹配上'}")
        # 唯一命名空间的标识：列表元素没有数据库约束，在事务内复查
        uniq = sorted({i for ref in p.new_nodes + list(p.upd_nodes) for i in nodes[ref]["props"].get("identifiers", [])
                       if NAMESPACES[i.partition(":")[0]]})
        if uniq:
            dup = tx.run("UNWIND $ids AS i MATCH (o) WHERE (o:Entity OR o:Concept) AND i IN o.identifiers "
                         "WITH i, collect(o.id) AS owners WHERE size(owners) > 1 RETURN i, owners", ids=uniq).data()
            if dup:
                raise RuntimeError(f"唯一标识重复（整批回滚）：{dup}")
        # 关系：按 (类型, 端点主 Label) 分组；语义对称的用无方向 MERGE
        groups = defaultdict(list)
        for a, t, c in p.new_rels:
            groups[(t, nodes[a]["family"], nodes[c]["family"])].append({"a": ref2id[a], "c": ref2id[c]})
        for (t, fa, fc), rows in groups.items():
            arrow = "-" if t in SYMMETRIC else "->"
            c = tx.run(f"UNWIND $rows AS row MATCH (a:{fa} {{id: row.a}}) MATCH (c:{fc} {{id: row.c}}) "
                       f"MERGE (a)-[r:{t}]{arrow}(c) RETURN count(r) AS c", rows=rows).single()["c"]
            assert c == len(rows), f"{t}：{len(rows)} 条里只写入 {c} 条（端点没匹配上）"

    with driver.session(database=NEO4J_DB) as s:
        s.execute_write(work)
    print(f"{b['name']}：写入 对象 {len(p.new_nodes) + len(p.upd_nodes)}，"
          f"NameKey {len(p.new_keys)}，关系 {len(p.new_rels)}")

    again = plan_seed(b)   # 复核：写完之后库内容应与本批完全一致
    assert again.empty and not again.conflicts, "复核失败：写入后仍有差异"
    print("  ✓ 复核通过")
    return {ref: again.target[ref] for ref in nodes}

### 4.1 时序预测种子 `tsf.yml`

第一次运行时应全部是"新建"。ETT 与 ETTh1–ETTm2 共用 `url:https://github.com/zhouhaoyi/ETDataset`，会作为查重候选列出；它们是种子作者有意分开建的数据集，不影响写入。

In [ ]:
ids = IdAllocator()
print("分配起点", {k: v for k, v in ids.last.items() if v} or "（空库）")

p_tsf = plan_seed(seeds["tsf.yml"])
show_plan(p_tsf)

In [ ]:
ref2id = {"tsf.yml": apply_plan(p_tsf, ids)}
library_status()

### 4.2 RAG 种子 `rag.yml`

两个文件没有共享节点，也没有跨文件关系。查重候选来自同一 url 下的多个数据集与模型（如 UltraDomain、OpenAI 的 API 模型）。

In [ ]:
p_rag = plan_seed(seeds["rag.yml"])
show_plan(p_rag)

In [ ]:
ref2id["rag.yml"] = apply_plan(p_rag, ids)
library_status()

### 4.3 重跑

同一批再 plan 一次：全部不变、没有查重候选（只有新对象需要查重），解析到的 id 与写入时一致。

In [ ]:
for name, b in seeds.items():
    again = plan_seed(b)
    assert again.empty and not again.conflicts and not again.candidates, name
    assert again.target == ref2id[name], f"{name}：重跑解析到的 id 与写入时不同"
    print(f"{name}：{len(again.same_nodes)} 个对象、{again.same_keys} 个精确键、{len(again.same_rels)} 条关系均已存在")

## 5. 失败路径：只 plan，不写库

下面几个小批次写在 notebook 里，不属于种子。每个批次只跑 plan；会被拒绝的批次再调用一次 `apply_plan`，以确认它确实不写库。

**(a) 别名已注册给另一个对象。** 新方法的 alias `DMS` 已经是 `Direct multi-step forecasting` 的精确键。plan 报 `key-taken`，apply 整批拒绝，库内节点数不变。外部可以选择换一个 alias、改用经确认的作用域，或者判定这其实就是已有对象。

In [ ]:
bad_alias = check_seed({"nodes": [{
    "ref": "m_demo", "labels": ["Concept", "Method"],
    "properties": {"name": "Demo linear forecaster", "aliases": ["DMS"],
                   "definition": "Demonstration only: a node whose alias collides with an existing method."}}]},
    "bad_alias（inline）")
before = q("MATCH (n) RETURN count(n) AS c")[0]["c"]
p_bad = plan_seed(bad_alias)
show_plan(p_bad)
assert apply_plan(p_bad, ids) == {} and q("MATCH (n) RETURN count(n) AS c")[0]["c"] == before

**(b) 唯一命名空间在批内重复。** 两个数据集带同一个 `arxiv` 标识，`check_seed` 读文件时就拒绝，不必查库。如果其中一个已经在库里，plan 会报 `identifier-taken`；种子里没有唯一命名空间的标识，这条路径要等论文入库后才会真正走到。

In [ ]:
try:
    check_seed({"nodes": [
        {"ref": f"d_demo{j}", "labels": ["Entity", "Dataset"],
         "properties": {"name": f"Demo dataset {j}", "identifiers": ["arxiv:0000.00000"],
                        "description": "Demonstration only."}} for j in (1, 2)]}, "dup_arxiv（inline）")
except AssertionError as e:
    print("拒绝：", e)

**(c) 非唯一标识的多重命中。** 一个新数据集也挂在 `zhouhaoyi/ETDataset` 下：库里已有 5 个对象持有这个 url。plan 把它们全部列为查重候选，没有冲突，所以 apply 会照常写入（这里不写）。它是这 5 个之一，还是同一仓库里的第 6 个数据集，要由外部判断；读取模式下的 Resolve 则会返回 `stage=id, status=ambiguous`，再与名称取交集（§2.4）。

In [ ]:
p_url = plan_seed(check_seed({"nodes": [{
    "ref": "d_demo", "labels": ["Entity", "Dataset"],
    "properties": {"name": "Demo ETT subset", "identifiers": ["url:https://github.com/zhouhaoyi/ETDataset"],
                   "description": "Demonstration only: a new dataset published in an existing repository."}}]},
    "same_url（inline）"))
show_plan(p_url)
assert not p_url.conflicts and len(p_url.candidates) == 1

**(d) 名称命中即同一对象。** 批次里写一个 `MSE`，换了 definition：`MSE|Metric|global` 已注册，plan 把它解析为库里的 MSE，列为"更新"并给出字段差异，而不是新建一个同名 Metric。要不要接受这个更新，由提交者看完差异后决定。

In [ ]:
p_mse = plan_seed(check_seed({"nodes": [{
    "ref": "met_mse", "labels": ["Concept", "Metric"],
    "properties": {"name": "MSE", "direction": "lower", "definition": "Demonstration only: a revised definition."}}]},
    "mse_update（inline）"))
show_plan(p_mse)
assert list(p_mse.upd_nodes) == ["met_mse"] and not p_mse.new_nodes

## 6. 库内结果

- 对象上不存 `aliases`；`Get` 读取时由 NameKey 装配：`raw` 与对象 `name` 不同的键就是 alias。
- 每个对象都有一条与自己 `name` 对应的 NameKey；NameKey 总数等于种子里规范化去重后的称呼数。
- 共用同一个 url 的对象组：这是之后检验 Resolve 多重命中的现成用例。

In [ ]:
expect_keys = sum(len(n["keys"]) for b in seeds.values() for n in b["nodes"].values())
checks = q("""
    CALL () { MATCH (n) WHERE n.aliases IS NOT NULL RETURN count(n) AS with_aliases_list }
    CALL () { MATCH (n:Entity|Concept) WHERE NOT EXISTS { MATCH (k:NameKey)-[:NAMES]->(n) WHERE k.raw = n.name }
              RETURN count(n) AS without_name_key }
    CALL () { MATCH (k:NameKey) RETURN count(k) AS name_keys }
    RETURN with_aliases_list, without_name_key, name_keys""")[0]
print(checks, "期望 NameKey", expect_keys)
assert checks == {"with_aliases_list": 0, "without_name_key": 0, "name_keys": expect_keys}

In [ ]:
# Get 视角：对象 + 由 NameKey 装配的 aliases
pd.DataFrame(q("""
    MATCH (n:Entity|Concept)
    OPTIONAL MATCH (k:NameKey)-[:NAMES]->(n) WHERE k.raw <> n.name
    WITH n, collect(k.raw) AS aliases
    RETURN n.id AS id, [l IN labels(n) WHERE NOT l IN ['Entity', 'Concept']][0] AS kind, n.name AS name, aliases
    ORDER BY id LIMIT 12"""))

In [ ]:
# 共用同一 url 的对象组
pd.DataFrame(q("""
    MATCH (n:Entity) UNWIND n.identifiers AS i
    WITH i, collect(n.id + ' ' + n.name) AS objects WHERE i STARTS WITH 'url:' AND size(objects) > 1
    RETURN i AS identifier, size(objects) AS n, objects ORDER BY i"""))

这里还没覆盖的：

- **Resolve（读取模式）**：id 级多重命中后与名称取交集，交集唯一则 resolved，否则保留 ambiguous（§2.4）。上面的 url 组可以直接作为用例。
- **`ambiguous` 隔离**：只在批量导入或事后核查发现既有冲突时出现，需要一个核查流程，种子入库走不到。
- **NameKey 的撤销与改指向**：删 alias、合并对象（Q4）。
- **规范化的 Unicode 版本**：设计写的是 15.1，Python 3.12 的 `unicodedata` 是 15.0，版本号已如实记进 `normalizer_ref`。对现有种子的 ASCII 名称没有影响。
- **Content 一侧**：`Material`、`FROM`、Experiment 与 ResultUnit 等论文入库时再做，依赖入库表单的剩余三项（实例范围、方法变体、条件槽）。

In [ ]:
library_status()
driver.close()